In [13]:
import socket
from datetime import datetime

ALLOWED_TARGETS = {"127.0.0.1", "localhost", "scanme.nmap.org"}

common_services = {
    21: "FTP",
    22: "SSH",
    23: "Telnet",
    25: "SMTP",
    53: "DNS",
    80: "HTTP",
    110: "POP3",
    143: "IMAP",
    443: "HTTPS",
    587: "SMTP Submission",
    8080: "HTTP Alternate",
}


def validate_scan_inputs(target, ports, timeout_seconds=0.5):
    if not isinstance(target, str) or not target.strip():
        raise ValueError("Target must be a non-empty string.")

    target = target.strip().lower()

    if target not in ALLOWED_TARGETS:
        raise ValueError(
            f"Target '{target}' is not allowed. Use one of: {sorted(ALLOWED_TARGETS)}"
        )

    if not isinstance(ports, (list, tuple, set)) or not ports:
        raise ValueError("Ports must be a non-empty list, tuple, or set of integers.")

    validated_ports = []

    for port in ports:
        if not isinstance(port, int):
            raise ValueError(f"Invalid port '{port}'. Ports must be integers.")

        if port < 1 or port > 65535:
            raise ValueError(f"Invalid port '{port}'. Ports must be between 1 and 65535.")

        validated_ports.append(port)

    if not isinstance(timeout_seconds, (int, float)):
        raise ValueError("Timeout must be a number.")

    if timeout_seconds <= 0:
        raise ValueError("Timeout must be greater than 0.")

    return target, sorted(set(validated_ports)), float(timeout_seconds)


def scan_port(target_ip, port, timeout_seconds=0.5):
    scanner = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    scanner.settimeout(timeout_seconds)

    try:
        result = scanner.connect_ex((target_ip, port))

        if result == 0:
            return "open"

        return "closed"

    except socket.timeout:
        return "closed"

    except OSError as error:
        return f"error: {error}"

    finally:
        scanner.close()


def run_port_scan(target="127.0.0.1", ports=None, timeout_seconds=0.5):
    if ports is None:
        ports = [21, 22, 80, 443]

    target, ports, timeout_seconds = validate_scan_inputs(target, ports, timeout_seconds)

    try:
        target_ip = socket.gethostbyname(target)
    except socket.gaierror:
        raise ValueError(f"Could not resolve target hostname: {target}")

    open_ports = []
    closed_ports = []
    error_ports = []

    start_time = datetime.now()

    print(f"Starting scan")
    print(f"Target: {target}")
    print(f"Resolved IP: {target_ip}")
    print(f"Ports: {ports}")
    print(f"Timeout: {timeout_seconds} seconds")
    print("-" * 50)

    for port in ports:
        service_name = common_services.get(port, "Unknown")
        status = scan_port(target_ip, port, timeout_seconds)

        if status == "open":
            open_ports.append(port)
            print(f"Port {port:<5} {service_name:<18} OPEN")

        elif status == "closed":
            closed_ports.append(port)
            print(f"Port {port:<5} {service_name:<18} CLOSED")

        else:
            error_ports.append((port, status))
            print(f"Port {port:<5} {service_name:<18} {status}")

    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()

    print("-" * 50)
    print(f"Open ports: {open_ports}")
    print(f"Closed ports: {closed_ports}")

    if error_ports:
        print(f"Errors: {error_ports}")

    print(f"Scan completed in {duration:.3f} seconds")

    return {
        "target": target,
        "target_ip": target_ip,
        "open_ports": open_ports,
        "closed_ports": closed_ports,
        "error_ports": error_ports,
        "duration": duration,
    }

In [14]:
localhost_results = run_port_scan(
    target="127.0.0.1",
    ports=[21, 22, 80, 443, 8080],
    timeout_seconds=0.5,
)

Starting scan
Target: 127.0.0.1
Resolved IP: 127.0.0.1
Ports: [21, 22, 80, 443, 8080]
Timeout: 0.5 seconds
--------------------------------------------------
Port 21    FTP                CLOSED
Port 22    SSH                CLOSED
Port 80    HTTP               CLOSED
Port 443   HTTPS              CLOSED
Port 8080  HTTP Alternate     OPEN
--------------------------------------------------
Open ports: [8080]
Closed ports: [21, 22, 80, 443]
Scan completed in 2.038 seconds


In [15]:
scanme_results = run_port_scan(
    target="scanme.nmap.org",
    ports=[22, 80, 443],
    timeout_seconds=1.0,
)

Starting scan
Target: scanme.nmap.org
Resolved IP: 45.33.32.156
Ports: [22, 80, 443]
Timeout: 1.0 seconds
--------------------------------------------------
Port 22    SSH                OPEN
Port 80    HTTP               OPEN
Port 443   HTTPS              CLOSED
--------------------------------------------------
Open ports: [22, 80]
Closed ports: [443]
Scan completed in 1.111 seconds


In [16]:
invalid_tests = [
    {"target": "", "ports": [80], "timeout_seconds": 0.5},
    {"target": "example.com", "ports": [80], "timeout_seconds": 0.5},
    {"target": "127.0.0.1", "ports": [0, 80], "timeout_seconds": 0.5},
    {"target": "127.0.0.1", "ports": ["80"], "timeout_seconds": 0.5},
    {"target": "127.0.0.1", "ports": [80], "timeout_seconds": -1},
]

for test in invalid_tests:
    try:
        run_port_scan(**test)
    except ValueError as error:
        print(f"Handled invalid input: {error}")


Handled invalid input: Target must be a non-empty string.
Handled invalid input: Target 'example.com' is not allowed. Use one of: ['127.0.0.1', 'localhost', 'scanme.nmap.org']
Handled invalid input: Invalid port '0'. Ports must be between 1 and 65535.
Handled invalid input: Invalid port '80'. Ports must be integers.
Handled invalid input: Timeout must be greater than 0.
